# run.ipynb — one-click pipeline for THIS pod bundle

Runs, in order and each in its **own fresh kernel**:

`setup.ipynb` → `prepare_training.ipynb` → `training.ipynb`

**Fail-fast**: if any cell in a child notebook errors, `nbclient` raises
`CellExecutionError` here and the chain stops — later notebooks never start,
exactly like an uncaught exception in a plain Python script. The partially
executed copy (including the traceback) is saved next to this file as
`executed_<name>.ipynb`, so you can open it and see the failing cell.

Notes:
- Child notebooks are read from **this bundle's folder**, so the per-VM
  `VM_NAME` baked in by `prepare_pods.ipynb` is used as-is. Nothing to edit here.
- Child output **streams live** into this notebook (per message, not per cell),
  so the multi-hour training cell keeps printing as it goes. If it truly goes
  quiet, cross-check `realtime_reader.ipynb` / the per-VM log.
- Re-running is safe: setup/prepare skip finished work, training resumes and
  re-claims combos through the shared coordinator.

In [ ]:
# nbclient/nbformat ship with JupyterLab on the pods -- this is a no-op there.
try:
    import nbclient, nbformat  # noqa: F401
except ImportError:
    %pip install -q nbclient nbformat
    import nbclient, nbformat  # noqa: F401
print('nbclient', nbclient.__version__, '| nbformat', nbformat.__version__)

In [ ]:
# Execute the pipeline. Stops at the FIRST error (CellExecutionError propagates).
import time
from pathlib import Path

import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

HERE = Path.cwd()
NOTEBOOKS = ['setup.ipynb', 'prepare_training.ipynb', 'training.ipynb']


class StreamingClient(NotebookClient):
    """Echo child stdout/stderr LIVE, per iopub message, instead of after the
    cell finishes. Without this the multi-hour training cell prints nothing
    here until it ends, which looks exactly like a hang/truncation."""

    def process_message(self, msg, cell, cell_index, **kwargs):
        if msg.get('header', {}).get('msg_type') == 'stream':
            print(msg.get('content', {}).get('text', ''), end='', flush=True)
        return super().process_message(msg, cell, cell_index, **kwargs)


for name in NOTEBOOKS:
    out_path = HERE / f'executed_{name}'
    print(f"\n{'=' * 70}\n==  {name}  (started {time.strftime('%Y-%m-%d %H:%M:%S')})\n{'=' * 70}",
          flush=True)
    nb = nbformat.read(HERE / name, as_version=4)
    client = StreamingClient(
        nb,
        timeout=None,                      # the training cell legitimately runs for hours
        kernel_name='python3',
        resources={'metadata': {'path': str(HERE)}},
    )
    try:
        client.execute()
    except CellExecutionError:
        print(f'\n!! {name} FAILED -- chain aborted; later notebooks were NOT run.\n'
              f'!! Open {out_path.name} to see the failing cell + traceback.', flush=True)
        raise
    finally:
        nbformat.write(nb, out_path)       # keep outputs either way (success or failure)
    print(f'==  {name} OK -> {out_path.name}', flush=True)

print('\nALL NOTEBOOKS DONE. Next: check_paralle.ipynb (coordination view), '
      'realtime_reader.ipynb (live log), eval.ipynb (after the whole sweep).')